# caliper acceptance (Playbook A)

Run this on a **fresh** Colab GPU runtime, once per architecture (T4, then
A100, then L4). It works through the scriptable steps of
[`docs/acceptance/manual-playbook.md`](https://github.com/mansoor-mamnoon/caliper/blob/main/docs/acceptance/manual-playbook.md),
then writes `report.md` (commit under `docs/acceptance/reports/`) and
`selftest-<arch>.json` (commit under `selftest-reports/`).

Steps needing `nvbandwidth` / `ncu` / a power cap (5-partial, 6, 7-`ncu`, 9
on non-thermal cards) are Playbook B -- skipped here and left `PENDING` in
the report.


## Bootstrap: clone + editable install (Rust core + dev/GPU extras)


In [ ]:
!curl -sSf https://sh.rustup.rs | sh -s -- -y >/dev/null && . $HOME/.cargo/env; \
apt-get -qq install -y nsight-systems-cli 2>/dev/null || true
!git clone -q https://github.com/mansoor-mamnoon/caliper /content/caliper
%cd /content/caliper
!. $HOME/.cargo/env && pip -q install -e '.[dev,sweep,parquet,triton]'
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!caliper --version


## Collect: version, doctor, selftest  (steps 1-4)


In [ ]:
import json, subprocess, datetime, pathlib
import torch
maj, minr = torch.cuda.get_device_capability(0)
ARCH = f'sm_{maj}{minr}'
DATE = datetime.date.today().isoformat()
OUT = pathlib.Path('/content/acceptance_out'); OUT.mkdir(exist_ok=True)

def run(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(cmd, '->', p.returncode)
    return p

version = run('caliper --version').stdout.strip()
doctor = run('caliper doctor --json')
(OUT / f'doctor-{ARCH}.json').write_text(doctor.stdout)
selftest = run('caliper selftest --full --json')
(OUT / f'selftest-{ARCH}.json').write_text(selftest.stdout)
st = json.loads(selftest.stdout)
print('selftest:', st['result'], '| coverage:', st['coverage'])


## On-device test tiers  (FR evidence)


In [ ]:
# l2 oracles, l4 unlocked reproducibility, l6 end-to-end -- from the clone
!pytest -m 'l2 or l4 or l6' -q 2>&1 | tail -40


## 20-cell corpus sweep  ->  validate  ->  submit --dry-run  (steps 11, 13)


In [ ]:
from caliper import sweep, submit
from caliper.api import validate_records
from caliper.corpus.kernels import gemm

grid = sweep('examples/acceptance-sweep.yaml',
             run_cell=lambda cell, cfg: gemm.run(cell, cfg).to_dict(),
             parquet=str(OUT / f'rows-{ARCH}.parquet'))
print(len(grid), 'rows')
vr = validate_records(OUT / f'rows-{ARCH}.parquet')
print('validate:', vr['ok'], vr['n_invalid'], 'invalid')

b = submit(OUT / f'rows-{ARCH}.parquet', out=str(OUT / 'bundle'))
vb = validate_records(OUT / 'bundle')
print('bundle validate:', vb['ok'])


## Regression + negative-validate fixtures  (steps 12, 14)


In [ ]:
TD = 'tests/testdata'
cmp = run(f'caliper compare --baseline {TD}/base.parquet --candidate {TD}/slow.parquet --fail-on-regression')
print('compare exit', cmp.returncode, '(expect 1)')
print(cmp.stdout)
neg = []
for bad in ('over_peak_row.parquet', 'bundle_missing_field', 'bundle_nonreproducing', 'bundle_slow_calibration'):
    r = run(f'caliper validate {TD}/{bad}')
    neg.append(r.returncode == 1)
    print(' ', bad, '-> exit', r.returncode, '(expect 1)')


## do_bench shim  (step 10)


In [ ]:
import triton.testing
from caliper import do_bench as caliper_do_bench
a = torch.randn(4096, 4096, device='cuda', dtype=torch.float16)
bmat = torch.randn(4096, 4096, device='cuda', dtype=torch.float16)
fn = lambda: torch.matmul(a, bmat)
triton_ms = triton.testing.do_bench(fn)                 # mean
caliper_ms = caliper_do_bench(fn, return_mode='mean')   # same reduction
delta = abs(caliper_ms - triton_ms) / triton_ms
print(f'triton {triton_ms:.3f} ms | caliper {caliper_ms:.3f} ms | delta {delta*100:.1f}%')


## Write the report


In [ ]:
TEMPLATE = '''# caliper acceptance report
- Arch / GPU: {arch} / (fill card)      - Host: Colab        - Date: {date}
- Tier: 1                                - caliper: {version}
- Tools present: (fill: nsys / ptxas / cuobjdump; ncu unavailable on Colab)

| Step | Expected | Measured | Pass? | Notes |
|------|----------|----------|-------|-------|
| 1 install clean | version prints | {version} | {p1} | |
| 2 doctor fields | verdict + fields | doctor-{arch}.json | {p2} | cross-check vs nvidia-smi -q |
| 3 selftest --full | PASS / reduced | {selftest} / {coverage} | {p3} | selftest-{arch}.json attached |
| 4 O1 linearity | slope in [0.97,1.03] | see selftest report | {p3} | |
| 5 L2 flush A/B | small >=2x, large <5% | | PENDING | needs --no-flush-l2 sizes |
| 6 nvbandwidth | +-5% | | PENDING | Playbook B |
| 7 cuBLAS vs ncu | ptxas regs only on Colab | | PARTIAL | ncu unavailable |
| 8 reproducibility | unlocked CoV <5% | see l4 tier output | {p8} | |
| 9 throttle | flagged + dropped | | PENDING | thermal fallback on T4 |
| 10 do_bench shim | +-3% vs caliper | {shim_delta:.1f}% | {p10} | |
| 11 sweep + resume | valid parquet | {rows} rows, valid={valid} | {p11} | kill+resume: fill by hand |
| 12 compare regression | exit 1 + spill delta | exit {cmp_exit} | {p12} | |
| 13 submit dry-run | valid bundle | {bundle_ok} | {p13} | |
| 14 negative validate | 4/4 rejected | {neg_pass}/4 | {p14} | |

## NFR results
NFR-5 (unlocked): see the l4 tier output.  NFR-6: (time a 200us bench).

## Deviations / triage
(fill: none, or describe + link issue)
'''

def tick(ok): return 'YES' if ok else 'NO'
report = TEMPLATE.format(
    arch=ARCH, date=DATE, version=version,
    selftest=st['result'], coverage=st['coverage'],
    p1=tick(bool(version)), p2=tick(doctor.returncode in (0, 1)),
    p3=tick(st['result'] in ('PASS', 'ERROR')),  # ERROR expected until the oracle runner lands
    p8='(fill from l4 tier)', shim_delta=delta * 100, p10=tick(delta < 0.03),
    rows=len(grid), valid=vr['ok'], p11=tick(vr['ok']),
    cmp_exit=cmp.returncode, p12=tick(cmp.returncode == 1),
    bundle_ok=vb['ok'], p13=tick(vb['ok']),
    neg_pass=sum(neg), p14=tick(all(neg)),
)
path = OUT / f'{ARCH}-colab-{DATE}.md'
path.write_text(report)
print(report)
print('\n--> commit', path, 'under docs/acceptance/reports/ and',
      OUT / f'selftest-{ARCH}.json', 'under selftest-reports/')
